# Kármán vortex street with the common MORFE API

Two-dimensional incompressible flow past a cylinder loses stability at $Re_c \approx 49$ through a Hopf bifurcation — the Kármán vortex street. This example reduces the 57 860-DOF Navier–Stokes model to a **single complex ODE**, the Stuart–Landau equation

$$\dot z_1 = \lambda z_1 + c_{101}\,z_1\eta' + c_{210}\,z_1|z_1|^2 + \dots$$

The Reynolds number rides along as a parametric coordinate $\eta' = 1/Re - 1/Re_0$, so one run at the expansion point describes the whole bifurcation neighbourhood. The committed output uses order 3. To reproduce the order-9 reference, change `order = 3` to `order = 9` and rerun the notebook.

Before the first run, initialise the example environment from the MORFEFerrite repository root with `julia --project=examples/12_karman_hopf -e 'using Pkg; Pkg.develop(path="."); Pkg.instantiate()'`. This selects MORFEFerrite from the current checkout and downloads MORFE from Julia's package registry.

The first cell activates and instantiates that environment. `NSE` is only a short name for the fluid backend used to describe the flow case; `build_model` and `parametrise` belong to the common API.

In [1]:
import Pkg
Pkg.activate(@__DIR__; io = devnull)
Pkg.instantiate(; io = devnull)
using MORFE, MORFEFerrite
const NSE = MORFEFerrite.FluidNavierStokes # for incompressible Navier-Stokes

MORFEFerrite.FluidNavierStokes

## 1. Describe the fluid case

Choose the polynomial expansion order directly: `3` is the quick demonstration and `9` is the reference calculation.

`fluid_model` reads the mesh and assembles everything the flow problem needs: the P2/P1 Taylor-Hood spaces and their boundary conditions, the Newton solve for the steady base flow, the operators of the system linearised about it, and the viscous pieces that carry the Reynolds dependence. `Re = 49.03` is the expansion point, just above the critical Reynolds number. The mesh is a Turek–Schäfer channel with a ⌀0.1 m cylinder; its diameter is the reference length in $\nu = D/Re$.

In [2]:
order = 3  # Change to 9 for the reference calculation.
case = NSE.fluid_model(joinpath(@__DIR__, "cylinder_flow.msh"); Re = 49.03)

Info    : Reading '/Users/tiago/Desktop/AM-TUM/Code/MORFEFerrite/MORFEFerrite.jl/examples/12_karman_hopf/cylinder_flow.msh'...
Info    : 18 entities
Info    : 6636 nodes
Info    : 13272 elements
Info    : Done reading '/Users/tiago/Desktop/AM-TUM/Code/MORFEFerrite/MORFEFerrite.jl/examples/12_karman_hopf/cylinder_flow.msh'


[ Info: Grid: 12943 cells, 6636 nodes
[ Info: DOFs: 59066 total


[ Info:   Velocity per cell : 12  (range 1:12)
[ Info:   Pressure per cell : 3 (range 13:15)
[ Info:   Free DOFs (steady state) : 57860
[ Info:   Free DOFs (DPIM)         : 57860  (inlet frozen)


[ Info: FEM: 57860 free DOFs (steady state), 57860 (DPIM)
[ Info:   Newton iter 1 (Re₀ = 49.03): ‖R‖ = 0.1002


[ Info:   Newton iter 2 (Re₀ = 49.03): ‖R‖ = 0.03774
[ Info:   Newton iter 3 (Re₀ = 49.03): ‖R‖ = 0.005448


[ Info:   Newton iter 4 (Re₀ = 49.03): ‖R‖ = 0.001838
[ Info:   Newton iter 5 (Re₀ = 49.03): ‖R‖ = 0.0001751


[ Info:   Newton iter 6 (Re₀ = 49.03): ‖R‖ = 7.552e-7
[ Info:   Newton iter 7 (Re₀ = 49.03): ‖R‖ = 5.865e-11


[ Info: Linear operators assembled (Re₀ = 49.03)
[ Info:   B₁ nnz = 1691298,  B₀ nnz = 1691298
[ Info:   B₁ rank check: 51224 of 57860 diagonal entries nonzero
[ Info: K_visc assembled: 1691298 nonzeros in free subspace


AssembledFluidModel (Ferrite P2/P1 Taylor-Hood)
  Re₀       : 49.03
  free DOFs : 57860 (steady state), 57860 (DPIM)
  operators : B₀ nnz=1691298  B₁ nnz=1691298  K_visc nnz=1160300
  base flow : ‖s₀‖∞ = 2.305993117642014

## 2. Select the Hopf pair

Which modes span the manifold is a modelling decision, so it is made here in the driver — the backend has no policy about it. The two master coordinates are the Hopf pair; every other computed mode is an off-manifold target used only for diagnostics.

The eigenproblem $-B_0\,y = \lambda B_1 y$ is solved by shift-invert ARPACK. Because the shift $\sigma$ is complex, ARPACK returns only the modes near $\sigma$ — a strongly oscillatory mode's conjugate sits near $\bar\sigma$ and is never computed. `solve_hopf_eigenproblem` appends those missing halves itself, which is exact for real $B_0$ and $B_1$, so `conjugate_index` names the true partner of `hopf_index`. `case.B[1]` and `case.B[2]` are $B_0$ and $B_1$: the assembled model keeps its linear operators in one tuple, in the same order MORFE's model wants them.

In [3]:
spectrum = NSE.solve_hopf_eigenproblem(-case.B[1], case.B[2];
    nev = 40, sigma_re = 3.0, sigma_im = 8.0)

master = [spectrum.hopf_index, spectrum.conjugate_index]
outer = setdiff(eachindex(spectrum.eigenvalues), master)
spectrum.eigenvalues[master] # print the Hopf pair

  Shift σ = 3.0 + 8.0im,  nev = 40,  ncv = 120,  n = 57860



    #   Re(λ)           Im(λ)           |λ|         


  ────────────────────────────────────────────────────
    1      +0.004029    +16.859170     16.859170
    2      -2.113876     -0.000000      2.113876


    3      -2.911776     +0.011415      2.911799
    4      -2.911776     -0.011415      2.911799
    5      -5.135423     +0.000000      5.135423
    6      -6.690747     +0.000005      6.690747
    7     -10.251990     +4.471935     11.184878
    8     -11.022533     +7.464033     13.311950
    9      -6.542148    +18.415390     19.542935
   10     -11.176325    +10.696573     15.470195
   11      -9.412933     +0.572271      9.430313
   12      -9.413069     -0.572398      9.430456
   13     -11.565875     +2.109541     11.756684
   14     -11.894098    +13.952487     18.334161
   15     -13.207551     +4.100003     13.829296
   16     -13.691153     +8.310084     16.015778
   17     -11.720046     -0.000408     11.720046
   18     -13.867732     +5.986819     15.104834
   19     -13.789959    +10.611149     17.399984
   20     -13.493065     +1.729825     13.603496
   21     -11.565939     -2.107543     11.756389
   22     -14.160896    +12.755380     19.058612
   23     -13.204521

2-element Vector{ComplexF64}:
 0.00402856110873584 + 16.85916985616479im
 0.00402856110873584 - 16.85916985616479im

## 3. Build the MORFE model

`build_model` converts the fluid case and its spectrum into MORFE's physics-independent model and spectral data. It computes the left eigenvectors of the master modes only — each costs its own adjoint factorisation — and conjugates the partner of a pair rather than solving for it, so the conjugate symmetry the reduction relies on holds by construction.

`scale = 1e-2` is a mode gauge: a uniform factor on both eigenvector sides, for conditioning. It changes what `W` and `R` mean, so coefficients from two gauges are not comparable — which is why it is written at the call site rather than hidden inside a default. The returned `meta` contains auxiliary backend information, including the conjugate permutation displayed below.

In [4]:
(; model, spectral, meta) = build_model(case, spectrum;
    master = master, outer = outer, expansion_order = order, scale = 1e-2)
meta.conjugate_permutation # print the conjugate permutation

3-element Vector{Int64}:
 2
 1
 3

## 4. Parametrise the invariant manifold

`parametrise` computes the polynomial manifold map `W` and its reduced dynamics `R` up to the chosen order. The resonance configuration keeps near-resonant monomials in complex normal form; `tol_relative = 0.1` judges each target on its own frequency scale, flagging a monomial whose detuning is under 10 % of that target's $|\lambda|$. `outer_targets = true` also measures the monomials against the off-manifold modes — a diagnostic, since the solve reads the master block alone.

Evaluating `R` at the end of the cell displays the reduced system: row 1 is the Stuart–Landau equation, row 2 its conjugate, and row 3 is $\dot\eta' = 0$, the frozen Reynolds coordinate. The Hopf bifurcation is supercritical when $\mathrm{Re}\,c_{210} < 0$.

In [5]:
W, R = parametrise(model, spectral, order;
    resonance = ResonanceConfig(style = :complex_normal_form, tol_relative = 0.1,
        outer_targets = true))
R # print reduced dynamics

┌ Info: conjugate_permutation is active — the following assumptions must hold:
│   1. Real-valued FOM: all matrices in model.linear_terms and all nonlinear/force terms must have real-valued entries (eltype <: Real or purely-real complex).
│   2. Each mode either comes in a complex conjugate pair with another mode, or is self-paired  meaning it has a real eigenvalue and a real-valued mode shape.
│   3. Eigenvalue conjugacy is necessary but NOT sufficient for paired modes; the eigenvectors must satisfy master_modes[:, perm[r]] = conj(master_modes[:, r]).
│   4. If external modes are present (N_EXT > 0): the same pairing rules apply to the external eigenvalues, encoded in the NVAR-length permutation.
└ Passing an incorrect permutation silently corrupts the parametrisation and reduced-dynamics.
┌ Info: Using optimised FEM path: cached per-element loop
└   n_fem_terms = 1


ReducedDynamics{2, 3, ComplexF64}(DensePolynomial{ComplexF64, 3, 2, Matrix{ComplexF64}}(ComplexF64[0.00402856110873584 + 16.85916985616479im 0.0 + 0.0im … 0.0 - 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.00402856110873584 - 16.85916985616479im … 2981.130007368557 - 95.30710608426244im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im], MultiindexSet{3}(StaticArraysCore.SVector{3, Int64}[[1, 0, 0], [0, 1, 0], [0, 0, 1], [2, 0, 0], [1, 1, 0], [1, 0, 1], [0, 2, 0], [0, 1, 1], [0, 0, 2], [3, 0, 0], [2, 1, 0], [2, 0, 1], [1, 2, 0], [1, 1, 1], [1, 0, 2], [0, 3, 0], [0, 2, 1], [0, 1, 2], [0, 0, 3]], [0, 0, 3, 9, 19]), [3, 3, 3]), 1)

## Optional: save the standard ROM files

The common saver writes `W`, `R`, their coefficient table, and a summary to the example's `results` directory. This cell is optional; the ROM is already available in memory after the previous cell. `validate.jl` reads the coefficient table it writes.

In [6]:
MORFE.save_rom(joinpath(@__DIR__, "results"), W, R;
    external_system = meta.external_system);